In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("day-27")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)

:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-43e68c20-c092-4331-b3d4-dae76386ac47;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 140ms :: artifacts dl 5ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

In [3]:
customers_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/customers.csv')
orders_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/orders.csv')
products_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/products.csv')

26/08/12 14:46:21 WARN CredentialProviderListFactory: Credentials option fs.s3a.aws.credentials.provider contains AWS v1 SDK entry com.amazonaws.auth.profile.ProfileCredentialsProvider; mapping to software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


**Problem 1** | Medium

Top 2 products per category by revenue

Join orders with products. For each category, find the top 2 products by total revenue generated. Return category, product_name, total_revenue, rank.

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Join orders with products
joined = (
    orders_df.alias('o')
    .join(
        products_df.alias('p'),
        on='product_id',
        how='inner'
    )
)

# 2. Calculate revenue for each order
revenue_df = (
    joined
    .withColumn(
        'revenue',
        F.col('o.unit_price') * F.col('o.quantity')
    )
)

# 3. Total revenue per category + product
product_revenue = (
    revenue_df
    .groupBy(
        F.col('p.category').alias('category'),
        F.col('p.product_name').alias('product_name')
    )
    .agg(
        F.sum('revenue').alias('total_revenue')
    )
)

# 4. Rank products within each category
window_spec = (
    Window
    .partitionBy('category')
    .orderBy(F.col('total_revenue').desc())
)

ranked = (
    product_revenue
    .withColumn(
        'rank',
        F.dense_rank().over(window_spec)
    )
)

# 5. Keep top 2 products per category
result = (
    ranked
    .filter(F.col('rank') <= 2)
    .select(
        'category',
        'product_name',
        'total_revenue',
        'rank'
    )
    .orderBy(
        'category',
        'rank'
    )
)

result.show(truncate=False)

+---------------+--------------------+-----------------+----+
|category       |product_name        |total_revenue    |rank|
+---------------+--------------------+-----------------+----+
|Electronics    |Laptop Pro 15       |15599.88         |1   |
|Electronics    |Monitor 27inch 4K   |5849.869999999999|2   |
|Furniture      |Office Chair Deluxe |4199.88          |1   |
|Furniture      |Standing Desk       |2399.96          |2   |
|Office Supplies|Whiteboard Large    |129.99           |1   |
|Office Supplies|Cable Management Kit|79.96            |2   |
+---------------+--------------------+-----------------+----+



**Problem 2** | Medium

Customers whose latest order was cancelled

For each customer, find their most recent order (by order_date). Return customers whose most recent order has status = "Cancelled".

In [5]:
w = Window.partitionBy("customer_id").orderBy(F.col("order_date").desc())

latest_orders = orders_df.withColumn("rn", F.row_number().over(w)) \
    .filter(F.col("rn") == 1)

latest_orders.filter(F.col("status") == "Cancelled") \
    .select("customer_id", "order_id", "order_date", "status") \
    .show()

+-----------+--------+----------+---------+
|customer_id|order_id|order_date|   status|
+-----------+--------+----------+---------+
|       C013|   O0083|2023-09-07|Cancelled|
|       C014|   O0084|2023-09-10|Cancelled|
|       C015|   O0085|2023-09-13|Cancelled|
|       C016|   O0086|2023-09-16|Cancelled|
+-----------+--------+----------+---------+



**Problem 3** | Hard

Customer cohort analysis

Group customers by the month they signed up (their "cohort"). For each cohort, calculate: number of customers, and the percentage who placed at least one order. This requires a left join plus conditional aggregation.

In [6]:
customers_with_cohort = customers_df.withColumn(
    "cohort_month", F.date_format("signup_date", "yyyy-MM")
)

customers_who_ordered = orders_df.select("customer_id").distinct() \
    .withColumn("has_order", F.lit(1))

cohort_status = customers_with_cohort.join(
    customers_who_ordered, on="customer_id", how="left"
).withColumn("has_order", F.coalesce(F.col("has_order"), F.lit(0)))

cohort_status.groupBy("cohort_month").agg(
    F.count("customer_id").alias("num_customers"),
    F.round(100.0 * F.sum("has_order") / F.count("customer_id"), 1).alias("pct_who_ordered")
).orderBy("cohort_month").show(20)

+------------+-------------+---------------+
|cohort_month|num_customers|pct_who_ordered|
+------------+-------------+---------------+
|     2020-04|            1|          100.0|
|     2020-06|            1|          100.0|
|     2020-07|            1|          100.0|
|     2020-08|            1|          100.0|
|     2020-09|            1|          100.0|
|     2020-11|            1|          100.0|
|     2020-12|            1|          100.0|
|     2021-01|            1|          100.0|
|     2021-02|            1|          100.0|
|     2021-03|            1|          100.0|
|     2021-04|            1|          100.0|
|     2021-05|            1|          100.0|
|     2021-06|            1|          100.0|
|     2021-07|            1|          100.0|
|     2021-08|            1|          100.0|
|     2021-10|            1|          100.0|
|     2021-12|            1|          100.0|
|     2022-01|            1|          100.0|
|     2022-02|            1|          100.0|
|     2022

**Problem 4** | Hard

Revenue concentration — top 20% of customers

Rank customers by total spend. Determine what percentage of total company revenue comes from the top 20% of customers (by count). This is the classic "Pareto / 80-20" interview question.

In [7]:
customer_spend = orders_df.groupBy("customer_id").agg(
    F.sum(F.col("unit_price") * F.col("quantity")).alias("total_spend")
).orderBy(F.col("total_spend").desc())

total_customers = customer_spend.count()
top_20_pct_count = max(1, round(total_customers * 0.20))

total_revenue = customer_spend.agg(F.sum("total_spend")).collect()[0][0]

top_customers_revenue = customer_spend.limit(top_20_pct_count) \
    .agg(F.sum("total_spend")).collect()[0][0]

pct_from_top = round(100.0 * top_customers_revenue / total_revenue, 1)

print(f"Total customers           : {total_customers}")
print(f"Top 20% count             : {top_20_pct_count}")
print(f"Total company revenue     : ${total_revenue:,.2f}")
print(f"Revenue from top 20%      : ${top_customers_revenue:,.2f}")
print(f"Percentage from top 20%   : {pct_from_top}%")

Total customers           : 25
Top 20% count             : 5
Total company revenue     : $44,382.68
Revenue from top 20%      : $13,609.45
Percentage from top 20%   : 30.7%


**Problem 5** | Hard

Gap analysis — missing order_ids in a sequence

Order IDs follow the pattern O0001, O0002, etc. Assuming they should be sequential with no gaps, find any missing order numbers between the minimum and maximum order_id in the dataset.

In [8]:
# Extract numeric part of order_id
numbered = orders_df.withColumn(
    "order_num", F.substring(F.col("order_id"), 2, 10).cast("integer")
)

min_num = numbered.agg(F.min("order_num")).collect()[0][0]
max_num = numbered.agg(F.max("order_num")).collect()[0][0]

# Generate the full expected sequence
expected = spark.range(min_num, max_num + 1).withColumnRenamed("id", "expected_num")

# Anti-join — expected numbers that don't appear in actual data
missing = expected.join(
    numbered.select("order_num"),
    expected["expected_num"] == numbered["order_num"],
    how="left_anti"
)

print(f"Order number range: {min_num} to {max_num}")
print(f"Missing order numbers: {missing.count()}")
missing.orderBy("expected_num").show(20)

Order number range: 1 to 100


Missing order numbers: 0


+------------+
|expected_num|
+------------+
+------------+



In [9]:
spark.stop()